# COMP3516 2026 Spring Group Project
## Multi-modal human activity recognition

In this task, we will utilize the OctoNet dataset developed by HKU AIoT Lab to build a multi-modal human activity recognition system. In the dataset, we use 5 activities (sitting, sleeping, walking, falling down and jumping), and we record each sample using 3 modalities: infrared array (IRA), WiFi CSI and IMU.

The training set size is 400, and the masked testing set has 63 samples.

### Task 1: Visualize the data (10 pt)
1. Please load one IRA sample of each class from the dataset and visualize it using inferno colormap. Include these images in your report. Specify the sample used.
2. Please select one CSI sample, and visualize the amplitude of the second link and the subcarrier of index 1. Look at the sample below. You need also specify the sample and include the image in your report.

![sample IRA image](output.png)
![sample CSI image](output_wifi.png)

In [1]:
# Task1 visualization — updated to match sample structure
import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("./data_sources")
OUT_DIR = Path("./output_visualizations")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES = {
    "sit":      "20240519162427225649.pickle",
    "walk":     "20240519163436130972.pickle",
    "sleep":    "20240518140039079953.pickle",
    "falldown": "20240521213146575725.pickle",
    "jump":     "20240521213452255777.pickle",
}

def load_sample(fname):
    p = DATA_DIR / fname
    with open(p, "rb") as f:
        return pickle.load(f)

def plot_ira_sample(sample, out_path, title=None, cmap="inferno", vmin=None, vmax=None):
    ira_nodes = sample.get("modality_data", {}).get("IRA", [])
    if not ira_nodes or ira_nodes[0] is None:
        raise ValueError("No IRA frames found in sample")
    frames = np.asarray(ira_nodes[0].get("frames", []))
    if frames.size == 0:
        raise ValueError("Empty IRA frames")
    # frames shape expected (T, H, W) — average (interpolated) over time into one image
    img = frames.mean(axis=0)
    plt.figure(figsize=(4,4))
    im = plt.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
    if title:
        plt.title(title)
    plt.axis("off")
    # Colorbar as legend for heatmap
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label("Temperature", rotation=270, labelpad=15)
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight", dpi=150)
    plt.close()
    print("Saved IRA image:", out_path)

def plot_wifi_amplitude(sample, out_path, link_idx=1, subcarrier_idx=1, title=None):
    wifi_nodes = sample.get("modality_data", {}).get("wifi", [])
    if not wifi_nodes or wifi_nodes[0] is None:
        raise ValueError("No WiFi frames found in sample")
    frames = np.asarray(wifi_nodes[0].get("frames", []))  # expected (T, links, subcarriers), complex OK
    if frames.size == 0:
        raise ValueError("Empty WiFi frames")
    amp = np.abs(frames)  # amplitude
    if amp.ndim != 3:
        raise ValueError("Unexpected wifi frames shape")
    if link_idx >= amp.shape[1] or subcarrier_idx >= amp.shape[2]:
        raise IndexError("link_idx or subcarrier_idx out of range")
    series = amp[:, link_idx, subcarrier_idx]
    plt.figure(figsize=(6,3))
    plt.plot(series, linewidth=1.2)
    plt.xlabel("Time frame")
    plt.ylabel("Amplitude")
    if title:
        plt.title(title)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print("Saved WiFi amplitude plot:", out_path)

# IRA visualizations
for cls, fname in SAMPLES.items():
    s = load_sample(fname)
    out = OUT_DIR / f"ira_{cls}.png"
    plot_ira_sample(s, out, title=f"IRA - {cls}")

# run WiFi amplitude for an example file
wifi_sample = load_sample(SAMPLES["sit"])
out_wifi = OUT_DIR / "wifi_amp_link2_subcarrier1.png"
plot_wifi_amplitude(wifi_sample, out_wifi, link_idx=1, subcarrier_idx=1,
                    title="CSI Amplitude Link 2, Subcarrier 1")

print("\nAll visualisations completed successfully.")

Saved IRA image: output_visualizations/ira_sit.png
Saved IRA image: output_visualizations/ira_walk.png
Saved IRA image: output_visualizations/ira_sleep.png
Saved IRA image: output_visualizations/ira_falldown.png
Saved IRA image: output_visualizations/ira_jump.png
Saved WiFi amplitude plot: output_visualizations/wifi_amp_link2_subcarrier1.png

All visualisations completed successfully.


### Task 2: Single modality classification (40 pt)
For each modality, build a single modality classifier to recognize the 5 activities. You can choose any model architecture you like, but you need to justify your choice in the report. 

- Marking criteria: 
    - results fully reproducible (10 pt)
    - output accuracy (30 pt)
        - for each modality, highest accuracy (top1) among the class, 10pt
        - top [2 to 20%] accuracy among the class, 9pt
        - top (20% to 50%] accuracy among the class, 8pt
        - top (50% to 70%] accuracy among the class, 6pt
        - rest 30%, runnable code, 5pt

In [ ]:
import os
import csv
import random
import pickle
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, Subset, DataLoader

# Configuration
SEED = 42
DATA_DIR = Path("data_sources")
CSV_FILE = Path("activity_masked.csv")
OUT_DIR = Path("output_visualizations") # Models saved here
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Determinism
def set_global_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_global_seed(SEED)

# Default shapes used for padding/filling
DEFAULT_SHAPES = {
    "IRA": (1, 24, 32),
    "wifi": (1, 2, 114),
    "imu": (1, 13, 17),
}

# Safe pickle loader
def safe_load_pickle(path):
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception as e:
        print(f"[load error] {path}: {e}")
        return None

# Dataset class (self-contained)
class sample_dataset(Dataset):
    def __init__(self, dataset_path, dataset_csv=CSV_FILE, read_len=None, validate=True):
        self.dataset_path = Path(dataset_path)
        self.csv_path = Path(dataset_csv)
        self.samples = []
        self.read_csv(read_len)
        if validate:
            self._validate_samples()

    def read_csv(self, read_len=None):
        self.samples = []
        with open(self.csv_path, "r", newline="") as f:
            reader = csv.reader(f)
            header = next(reader, None)
            for row in reader:
                if len(row) < 3:
                    continue
                raw_id = (row[2] or "").strip()
                if raw_id in ("", "nan"):
                    continue
                # store as (filename, activity_name, activity_id)
                self.samples.append(row)
        if read_len is not None and read_len < len(self.samples):
            self.samples = random.sample(self.samples, read_len)

    def _validate_samples(self):
        kept = []
        for row in self.samples:
            p = Path(self.dataset_path) / row[0]
            if not p.exists():
                print(f"[missing] {row[0]}")
                continue
            data = safe_load_pickle(p)
            if data is None or not isinstance(data, dict) or "modality_data" not in data:
                print(f"[invalid] {row[0]}")
                continue
            kept.append(row)
        print(f"Validated dataset: {len(kept)} / {len(self.samples)} retained")
        self.samples = kept

    def _pad_or_fill_frames(self, frames, default_shape, use_abs=False):
        arr = np.asarray(frames)
        if arr.size == 0:
            return np.zeros(default_shape, dtype=np.float32)
        if use_abs:
            arr = np.abs(arr)
        return arr.astype(np.float32)

    def _normalize_sample(self, data_dict):
        modality_data = data_dict.get("modality_data", {})
        for modality, default_shape in DEFAULT_SHAPES.items():
            node_list = modality_data.get(modality)
            if not node_list:
                modality_data[modality] = [{"frames": np.zeros(default_shape, dtype=np.float32)}]
                continue
            normalized_nodes = []
            for node_item in node_list:
                if node_item is None:
                    normalized_nodes.append({"frames": np.zeros(default_shape, dtype=np.float32)})
                    continue
                normalized_node = dict(node_item)
                normalized_node["frames"] = self._pad_or_fill_frames(
                    node_item.get("frames", []),
                    default_shape,
                    use_abs=(modality == "wifi"),
                )
                normalized_nodes.append(normalized_node)
            modality_data[modality] = normalized_nodes
        data_dict["modality_data"] = modality_data
        return data_dict

    @staticmethod
    def collate_fn(batch):
        modalities = ("IRA", "wifi", "imu")
        default_shapes = DEFAULT_SHAPES
        raw_samples, labels = zip(*batch)
        samples = [dict(s) for s in raw_samples]

        per_modality = {m: [] for m in modalities}
        max_time = {m: 1 for m in modalities}

        for sample in samples:
            modality_data = sample.get("modality_data", {})
            for modality in modalities:
                node_list = modality_data.get(modality, [])
                if not node_list or node_list[0] is None:
                    frames = np.zeros(default_shapes[modality], dtype=np.float32)
                else:
                    frames = np.asarray(node_list[0].get("frames", []))
                    if frames.size == 0:
                        frames = np.zeros(default_shapes[modality], dtype=np.float32)
                    else:
                        if modality == "wifi":
                            frames = np.abs(frames)
                        frames = frames.astype(np.float32)
                per_modality[modality].append(frames)
                max_time[modality] = max(max_time[modality], frames.shape[0])

        batch_modality_data = {}
        for modality in modalities:
            padded = []
            target_t = max_time[modality]
            for frames in per_modality[modality]:
                if frames.shape[0] < target_t:
                    pad_shape = (target_t - frames.shape[0], *frames.shape[1:])
                    pad = np.zeros(pad_shape, dtype=np.float32)
                    frames = np.concatenate([frames, pad], axis=0)
                padded.append(torch.from_numpy(frames))
            batch_modality_data[modality] = torch.stack(padded, dim=0)  # [B, T, ...]
        labels_tensor = torch.tensor([int(x) for x in labels], dtype=torch.long)
        user_ids = [s.get("user_id") for s in samples]

        # metadata (without frames)
        metadata_samples = []
        for sample in samples:
            sample_meta = {}
            for key, value in sample.items():
                if key != "modality_data":
                    sample_meta[key] = value
            modality_meta = {}
            for modality, node_list in sample.get("modality_data", {}).items():
                node_meta_list = []
                for node_item in node_list:
                    if node_item is None:
                        node_meta_list.append(None)
                    else:
                        node_meta = {k: v for k, v in node_item.items() if k != "frames"}
                        node_meta_list.append(node_meta)
                modality_meta[modality] = node_meta_list
            sample_meta.update(modality_meta)
            metadata_samples.append(sample_meta)

        return {
            "metadata": metadata_samples,
            "user_id": user_ids,
            "modality_data": batch_modality_data,
            "label": labels_tensor,
        }

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row = self.samples[idx]
        data_pickle_path = Path(self.dataset_path) / row[0]
        with open(data_pickle_path, "rb") as f:
            data_dict = pickle.load(f)
        data_dict = self._normalize_sample(data_dict)
        data_label = row[2]
        return data_dict, data_label

# Build dataset and dataloaders
set_global_seed(SEED)
dataset_obj = sample_dataset(DATA_DIR, dataset_csv=CSV_FILE, read_len=None, validate=True)
n = len(dataset_obj)
if n == 0:
    raise RuntimeError("No valid labeled samples found in dataset. Fix dataset or validation flag.")

train_count = int(0.8 * n)
train_indices = list(range(train_count))
valid_indices = list(range(train_count, n))

# worker seed for deterministic dataloader workers
def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    Subset(dataset_obj, train_indices),
    batch_size=16,
    shuffle=True,
    collate_fn=sample_dataset.collate_fn,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)
valid_loader = DataLoader(
    Subset(dataset_obj, valid_indices),
    batch_size=32,
    shuffle=False,
    collate_fn=sample_dataset.collate_fn,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)

print(f"Dataset: {n} labeled samples; train={len(train_indices)} valid={len(valid_indices)}")

# Define simple classifier model
class TemporalLSTMClassifier(nn.Module):
    """
    LSTM-based classifier for a single modality.
    Input shape from dataloader: (batch, time, height, width) or (batch, time, channels, height)
    It first flattens the spatial dimensions at each time step,
    then passes the sequence through an LSTM, and finally classifies using the last hidden state.
    """
    def __init__(self, input_shape, hidden_size=128, num_layers=2, dropout=0.2):
        """
        input_shape: tuple (H, W) or (C, H) - the spatial dimensions after temporal mean removal.
                     For IRA: (24,32); for WiFi: (2,114); for IMU: (13,17)
        hidden_size: LSTM hidden dimension
        num_layers: number of LSTM layers
        dropout: dropout between LSTM layers (if num_layers > 1)
        """
        super().__init__()
        self.input_shape = input_shape
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Flatten spatial dimensions for each time step
        flat_dim = int(np.prod(input_shape))
        
        # Optional projection layer to reduce dimension before LSTM
        self.proj = nn.Linear(flat_dim, hidden_size)
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.classifier = nn.Linear(hidden_size, 5)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        """
        x: tensor of shape (batch, time, *spatial_dims)
        """
        batch_size, time_steps = x.shape[0], x.shape[1]
        
        # Flatten spatial dimensions: (batch, time, flat_dim)
        x = x.view(batch_size, time_steps, -1)
        
        # Project to hidden_size
        x = self.proj(x)                     # (batch, time, hidden_size)
        x = self.dropout(x)
        
        # LSTM
        out, (h_n, c_n) = self.lstm(x)       # out: (batch, time, hidden_size)
        
        last_hidden = h_n[-1]                # (batch, hidden_size)
        
        # Classify
        logits = self.classifier(last_hidden)
        return logits

# Helpers for data extraction and training/eval
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_modality_input(batch, modality):
    x = batch["modality_data"][modality].float()  # [B, T, ...]
    if modality == "wifi":
        x = x.abs()  
    return x   #

# Evulate Model
def evaluate_modality(model, loader, modality, device):
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            inputs = get_modality_input(batch, modality).to(device)
            labels = batch["label"].to(device) - 1  # CSV labels are 1..5 -> convert 0..4
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total if total > 0 else 0.0

def train_modality(model, train_loader, valid_loader, modality, epochs=15, lr=1e-3, device="cpu"):
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_acc = 0.0
    best_state = None
    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_total = 0
        for batch in train_loader:
            inputs = get_modality_input(batch, modality).to(device)
            labels = batch["label"].to(device) - 1
            opt.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            opt.step()
            running_loss += loss.item() * labels.size(0)
            running_total += labels.size(0)
        train_loss = running_loss / running_total if running_total > 0 else 0.0
        val_acc = evaluate_modality(model, valid_loader, modality, device)
        print(f"[{modality}] Epoch {ep}/{epochs} train_loss={train_loss:.4f} val_acc={val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc

# Input shapes after mean over time
ira_shape = (24, 32)
wifi_shape = (2, 114)
imu_shape = (13, 17)

# Instantiate and train per modality
ira_model = TemporalLSTMClassifier(ira_shape, hidden_size=128, num_layers=2, dropout=0.2)
wifi_model = TemporalLSTMClassifier(wifi_shape, hidden_size=128, num_layers=2, dropout=0.2)
imu_model = TemporalLSTMClassifier(imu_shape, hidden_size=128, num_layers=2, dropout=0.2)

print("Training IRA model...")
ira_model, ira_acc = train_modality(ira_model, train_loader, valid_loader, "IRA", epochs=50, lr=1e-3, device=device)
print("Training WiFi model...")
wifi_model, wifi_acc = train_modality(wifi_model, train_loader, valid_loader, "wifi", epochs=50, lr=1e-3, device=device)
print("Training IMU model...")
imu_model, imu_acc = train_modality(imu_model, train_loader, valid_loader, "imu", epochs=50, lr=1e-3, device=device)

print("\nValidation accuracies -> IRA: {:.4f}, WiFi: {:.4f}, IMU: {:.4f}".format(ira_acc, wifi_acc, imu_acc))

# Save models
torch.save(ira_model.state_dict(), OUT_DIR / "ira_model.pth")
torch.save(wifi_model.state_dict(), OUT_DIR / "wifi_model.pth")
torch.save(imu_model.state_dict(), OUT_DIR / "imu_model.pth")
print(f"Saved models in {OUT_DIR}")

print("Task 2 finished.")

Validated dataset: 400 / 400 retained
Dataset: 400 labeled samples; train=320 valid=80
Training IRA model...
[IRA] Epoch 1/50 train_loss=1.5752 val_acc=0.2625
[IRA] Epoch 2/50 train_loss=1.4849 val_acc=0.2500
[IRA] Epoch 3/50 train_loss=1.4615 val_acc=0.2625
[IRA] Epoch 4/50 train_loss=1.4524 val_acc=0.2625
[IRA] Epoch 5/50 train_loss=1.4273 val_acc=0.2375
[IRA] Epoch 6/50 train_loss=1.4322 val_acc=0.3375
[IRA] Epoch 7/50 train_loss=1.3987 val_acc=0.3875
[IRA] Epoch 8/50 train_loss=1.4766 val_acc=0.2500
[IRA] Epoch 9/50 train_loss=1.4407 val_acc=0.3125
[IRA] Epoch 10/50 train_loss=1.4415 val_acc=0.2500
[IRA] Epoch 11/50 train_loss=1.4180 val_acc=0.2500
[IRA] Epoch 12/50 train_loss=1.3739 val_acc=0.3250
[IRA] Epoch 13/50 train_loss=1.4582 val_acc=0.2750
[IRA] Epoch 14/50 train_loss=1.4665 val_acc=0.2750
[IRA] Epoch 15/50 train_loss=1.4077 val_acc=0.3000
[IRA] Epoch 16/50 train_loss=1.3996 val_acc=0.3000
[IRA] Epoch 17/50 train_loss=1.4008 val_acc=0.3250
[IRA] Epoch 18/50 train_loss=1.41

Predict and Export CSV

In [14]:
import csv
import pickle
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

# ========== CONFIGURATION (adjust paths as needed) ==========
DATA_DIR = Path("data_sources")          # folder with pickle files
CSV_FILE = Path("activity_masked.csv")   # original CSV with some labels
OUT_DIR = Path("output_visualizations")  # where models are saved
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class sample_dataset(Dataset):
    def __init__(self, dataset_path, dataset_csv=CSV_FILE, read_len=None, validate=True):
        self.dataset_path = Path(dataset_path)
        self.csv_path = Path(dataset_csv)
        self.samples = []
        self.read_csv(read_len)
        if validate:
            self._validate_samples()

    def read_csv(self, read_len=None):
        self.samples = []
        with open(self.csv_path, "r", newline="") as f:
            reader = csv.reader(f)
            header = next(reader, None)
            for row in reader:
                if len(row) < 3:
                    continue
                raw_id = (row[2] or "").strip()
                if raw_id in ("", "nan"):
                    continue
                # store as (filename, activity_name, activity_id)
                self.samples.append(row)
        if read_len is not None and read_len < len(self.samples):
            self.samples = random.sample(self.samples, read_len)

    def _validate_samples(self):
        kept = []
        for row in self.samples:
            p = Path(self.dataset_path) / row[0]
            if not p.exists():
                print(f"[missing] {row[0]}")
                continue
            data = safe_load_pickle(p)
            if data is None or not isinstance(data, dict) or "modality_data" not in data:
                print(f"[invalid] {row[0]}")
                continue
            kept.append(row)
        print(f"Validated dataset: {len(kept)} / {len(self.samples)} retained")
        self.samples = kept

    def _pad_or_fill_frames(self, frames, default_shape, use_abs=False):
        arr = np.asarray(frames)
        if arr.size == 0:
            return np.zeros(default_shape, dtype=np.float32)
        if use_abs:
            arr = np.abs(arr)
        return arr.astype(np.float32)

    def _normalize_sample(self, data_dict):
        modality_data = data_dict.get("modality_data", {})
        for modality, default_shape in DEFAULT_SHAPES.items():
            node_list = modality_data.get(modality)
            if not node_list:
                modality_data[modality] = [{"frames": np.zeros(default_shape, dtype=np.float32)}]
                continue
            normalized_nodes = []
            for node_item in node_list:
                if node_item is None:
                    normalized_nodes.append({"frames": np.zeros(default_shape, dtype=np.float32)})
                    continue
                normalized_node = dict(node_item)
                normalized_node["frames"] = self._pad_or_fill_frames(
                    node_item.get("frames", []),
                    default_shape,
                    use_abs=(modality == "wifi"),
                )
                normalized_nodes.append(normalized_node)
            modality_data[modality] = normalized_nodes
        data_dict["modality_data"] = modality_data
        return data_dict

    @staticmethod
    def collate_fn(batch):
        modalities = ("IRA", "wifi", "imu")
        default_shapes = DEFAULT_SHAPES
        raw_samples, labels = zip(*batch)
        samples = [dict(s) for s in raw_samples]

        per_modality = {m: [] for m in modalities}
        max_time = {m: 1 for m in modalities}

        for sample in samples:
            modality_data = sample.get("modality_data", {})
            for modality in modalities:
                node_list = modality_data.get(modality, [])
                if not node_list or node_list[0] is None:
                    frames = np.zeros(default_shapes[modality], dtype=np.float32)
                else:
                    frames = np.asarray(node_list[0].get("frames", []))
                    if frames.size == 0:
                        frames = np.zeros(default_shapes[modality], dtype=np.float32)
                    else:
                        if modality == "wifi":
                            frames = np.abs(frames)
                        frames = frames.astype(np.float32)
                per_modality[modality].append(frames)
                max_time[modality] = max(max_time[modality], frames.shape[0])

        batch_modality_data = {}
        for modality in modalities:
            padded = []
            target_t = max_time[modality]
            for frames in per_modality[modality]:
                if frames.shape[0] < target_t:
                    pad_shape = (target_t - frames.shape[0], *frames.shape[1:])
                    pad = np.zeros(pad_shape, dtype=np.float32)
                    frames = np.concatenate([frames, pad], axis=0)
                padded.append(torch.from_numpy(frames))
            batch_modality_data[modality] = torch.stack(padded, dim=0)  # [B, T, ...]
        labels_tensor = torch.tensor([int(x) for x in labels], dtype=torch.long)
        user_ids = [s.get("user_id") for s in samples]

        # metadata (without frames)
        metadata_samples = []
        for sample in samples:
            sample_meta = {}
            for key, value in sample.items():
                if key != "modality_data":
                    sample_meta[key] = value
            modality_meta = {}
            for modality, node_list in sample.get("modality_data", {}).items():
                node_meta_list = []
                for node_item in node_list:
                    if node_item is None:
                        node_meta_list.append(None)
                    else:
                        node_meta = {k: v for k, v in node_item.items() if k != "frames"}
                        node_meta_list.append(node_meta)
                modality_meta[modality] = node_meta_list
            sample_meta.update(modality_meta)
            metadata_samples.append(sample_meta)

        return {
            "metadata": metadata_samples,
            "user_id": user_ids,
            "modality_data": batch_modality_data,
            "label": labels_tensor,
        }

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row = self.samples[idx]
        data_pickle_path = Path(self.dataset_path) / row[0]
        with open(data_pickle_path, "rb") as f:
            data_dict = pickle.load(f)
        data_dict = self._normalize_sample(data_dict)
        data_label = row[2]
        return data_dict, data_label


set_global_seed(SEED)
dataset_obj = sample_dataset(DATA_DIR, dataset_csv=CSV_FILE, read_len=None, validate=True)
n = len(dataset_obj)
if n == 0:
    raise RuntimeError("No valid labeled samples found in dataset. Fix dataset or validation flag.")

# Default shapes for each modality (same as in training)
DEFAULT_SHAPES = {
    "IRA": (1, 24, 32),
    "wifi": (1, 2, 114),
    "imu": (1, 13, 17),
}

# ========== HELPER FUNCTIONS (replicated from training) ==========
def safe_load_pickle(path):
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception as e:
        print(f"[load error] {path}: {e}")
        return None

def normalize_sample(data_dict):
    """Replicates the dataset's _normalize_sample method for a single file."""
    modality_data = data_dict.get("modality_data", {})
    for modality, default_shape in DEFAULT_SHAPES.items():
        node_list = modality_data.get(modality)
        if not node_list:
            modality_data[modality] = [{"frames": np.zeros(default_shape, dtype=np.float32)}]
            continue
        normalized_nodes = []
        for node_item in node_list:
            if node_item is None:
                normalized_nodes.append({"frames": np.zeros(default_shape, dtype=np.float32)})
                continue
            normalized_node = dict(node_item)
            frames = np.asarray(node_item.get("frames", []))
            if frames.size == 0:
                frames = np.zeros(default_shape, dtype=np.float32)
            if modality == "wifi":
                frames = np.abs(frames)
            normalized_node["frames"] = frames.astype(np.float32)
            normalized_nodes.append(normalized_node)
        modality_data[modality] = normalized_nodes
    data_dict["modality_data"] = modality_data
    return data_dict

# ========== MODEL DEFINITION (must match training architecture) ==========
class TemporalLSTMClassifier(nn.Module):
    def __init__(self, input_shape, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        flat_dim = int(np.prod(input_shape))
        self.proj = nn.Linear(flat_dim, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.classifier = nn.Linear(hidden_size, 5)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, time, *spatial_dims)
        batch, time = x.shape[0], x.shape[1]
        x = x.view(batch, time, -1)
        x = self.proj(x)
        x = self.dropout(x)
        out, (h_n, _) = self.lstm(x)
        last_hidden = h_n[-1]
        return self.classifier(last_hidden)

# ========== PREDICTION FUNCTION ==========
def predict_single_file(model, modality, file_path):
    data = safe_load_pickle(file_path)
    if data is None:
        return None
    data = normalize_sample(data)
    node_list = data["modality_data"].get(modality, [])
    if not node_list or node_list[0] is None:
        arr = np.zeros(DEFAULT_SHAPES[modality], dtype=np.float32)
    else:
        arr = np.asarray(node_list[0].get("frames", []))
        if arr.size == 0:
            arr = np.zeros(DEFAULT_SHAPES[modality], dtype=np.float32)
    # arr shape: (T, H, W) or (T, C, H) – keep time dimension
    t = torch.from_numpy(arr).unsqueeze(0).float().to(DEVICE)  # (1, T, ...)
    with torch.no_grad():
        logits = model(t)
        pred = int(logits.argmax(dim=1).item()) + 1   # convert 0‑4 → 1‑5
    return pred

def load_model(modality, input_shape, model_path):
    model = TemporalLSTMClassifier(input_shape, hidden_size=128, num_layers=2, dropout=0.2)
    state = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    return model

# ========== LOAD MODELS ==========
ira_model = load_model("IRA", (24,32), OUT_DIR / "ira_model.pth")
wifi_model = load_model("wifi", (2,114), OUT_DIR / "wifi_model.pth")
imu_model = load_model("imu", (13,17), OUT_DIR / "imu_model.pth")

# ========== READ ORIGINAL CSV ==========
rows = []
with open(CSV_FILE, "r", newline="") as f:
    reader = csv.reader(f)
    header = next(reader, None)   # keep header if present
    rows = list(reader)

# ========== WRITE PREDICTION CSVS ==========
def write_pred_csv(model, modality, out_name):
    out_path = Path(out_name)
    out_rows = []
    out_header = ["filename", "activity", "activity_id"]
    for row in rows:
        if len(row) < 3:
            row = row + [""] * (3 - len(row))
        fname = row[0]
        label = (row[2] or "").strip()
        if label == "" or label == "nan":
            p = DATA_DIR / fname
            pred = predict_single_file(model, modality, p)
            if pred is None:
                out_rows.append([fname, "", ""])
            else:
                act_name = {
                    "1": "sitting", "2": "walking", "3": "sleeping",
                    "4": "falldown", "5": "jumping"
                }.get(str(pred), "")
                out_rows.append([fname, act_name, str(pred)])
        else:
            out_rows.append(row)   # keep original ground truth
    with open(out_path, "w", newline="") as wf:
        writer = csv.writer(wf)
        writer.writerow(out_header)
        writer.writerows(out_rows)
    print(f"Wrote {out_path} ({modality})")

# Generate predictions for all three modalities
write_pred_csv(ira_model, "IRA", "activity_ira.csv")
write_pred_csv(wifi_model, "wifi", "activity_wifi.csv")
write_pred_csv(imu_model, "imu", "activity_imu.csv")

Validated dataset: 400 / 400 retained
Wrote activity_ira.csv (IRA)
Wrote activity_wifi.csv (wifi)
Wrote activity_imu.csv (imu)


### Task 3: Multi modality classification (10 pt)
Build a multi modality classifier to recognize the 5 activities.

- Marking criteria: 
    - Performance improvement over your group's best single modality classifiers (7 pt)
    - Performance improvement over your group's best single modality classifiers by at least 10% (10 pt)

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modalities and spatial shapes used in previous cells
modalities = ["IRA", "wifi", "imu"]
modality_shapes = {"IRA": (24, 32), "wifi": (2, 114), "imu": (13, 17)}

class ModalEncoder(nn.Module):
    def __init__(self, input_shape, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        flat_dim = int(input_shape[0] * input_shape[1])
        self.proj = nn.Linear(flat_dim, hidden_size)
        self.lstm = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: [B, T, ...spatial...]
        b, t = x.shape[0], x.shape[1]
        x = x.view(b, t, -1)            # (B, T, flat_dim)
        x = self.proj(x)                # (B, T, hidden)
        x = self.dropout(x)
        _, (h_n, _) = self.lstm(x)      # h_n: (num_layers, B, hidden)
        return h_n[-1]                  # (B, hidden)

class MultiModalLSTMClassifier(nn.Module):
    def __init__(self, modality_shapes, hidden_size=128, num_layers=2, dropout=0.2, num_classes=5):
        super().__init__()
        self.modalities = list(modality_shapes.keys())
        self.encoders = nn.ModuleDict({
            m: ModalEncoder(modality_shapes[m], hidden_size, num_layers, dropout)
            for m in self.modalities
        })
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * len(self.modalities), hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, batch_modality_dict):
        feats = []
        for m in self.modalities:
            x = batch_modality_dict[m]
            if m == "wifi":
                x = x.abs()
            feats.append(self.encoders[m](x))
        fused = torch.cat(feats, dim=1)
        return self.classifier(fused)

def evaluate_multi(model, loader, device):
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            inputs = {m: batch["modality_data"][m].float().to(device) for m in modalities}
            labels = batch["label"].to(device) - 1
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total if total > 0 else 0.0

def train_multi(model, train_loader, valid_loader, epochs=30, lr=1e-3, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_acc = 0.0
    best_state = None
    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_total = 0
        for batch in train_loader:
            inputs = {m: batch["modality_data"][m].float().to(device) for m in modalities}
            labels = batch["label"].to(device) - 1
            opt.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            opt.step()
            running_loss += loss.item() * labels.size(0)
            running_total += labels.size(0)
        train_loss = running_loss / running_total if running_total > 0 else 0.0
        val_acc = evaluate_multi(model, valid_loader, device)
        print(f"[multi] Epoch {ep}/{epochs} train_loss={train_loss:.4f} val_acc={val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc

# ---- Instantiate and train ----
# Ensure SEED, train_loader, valid_loader, OUT_DIR are defined in the notebook.
try:
    set_global_seed(SEED)
except Exception:
    pass

multi_model = MultiModalLSTMClassifier(modality_shapes, hidden_size=128, num_layers=2, dropout=0.2)
multi_model, multi_acc = train_multi(multi_model, train_loader, valid_loader, epochs=50, lr=1e-3, device=device)
print(f"Multi-modality validation accuracy: {multi_acc:.4f}")

try:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    pass
torch.save(multi_model.state_dict(), (OUT_DIR / "multi_model.pth"))
print("Saved multi-model to", OUT_DIR / "multi_model.pth")

[multi] Epoch 1/50 train_loss=1.5872 val_acc=0.2375
[multi] Epoch 2/50 train_loss=1.5021 val_acc=0.3000
[multi] Epoch 3/50 train_loss=1.4476 val_acc=0.2375
[multi] Epoch 4/50 train_loss=1.4200 val_acc=0.4125
[multi] Epoch 5/50 train_loss=1.4427 val_acc=0.2375
[multi] Epoch 6/50 train_loss=1.3874 val_acc=0.2500
[multi] Epoch 7/50 train_loss=1.3780 val_acc=0.2625
[multi] Epoch 8/50 train_loss=1.5110 val_acc=0.2500
[multi] Epoch 9/50 train_loss=1.4578 val_acc=0.3375
[multi] Epoch 10/50 train_loss=1.3923 val_acc=0.2750
[multi] Epoch 11/50 train_loss=1.4302 val_acc=0.2625
[multi] Epoch 12/50 train_loss=1.4054 val_acc=0.2500
[multi] Epoch 13/50 train_loss=1.4004 val_acc=0.2750
[multi] Epoch 14/50 train_loss=1.3982 val_acc=0.2875
[multi] Epoch 15/50 train_loss=1.3705 val_acc=0.2500
[multi] Epoch 16/50 train_loss=1.3635 val_acc=0.2875
[multi] Epoch 17/50 train_loss=1.3961 val_acc=0.2750
[multi] Epoch 18/50 train_loss=1.3434 val_acc=0.2500
[multi] Epoch 19/50 train_loss=1.3128 val_acc=0.3750
[m

Predict and Export CSV (MultiModal)

In [22]:
# Multimodal prediction + CSV export (paste into your notebook)
import csv
import pickle
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

DATA_DIR = Path("data_sources")
CSV_FILE = Path("activity_masked.csv")
OUT_DIR = Path("output_visualizations")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Default shapes (same as training)
DEFAULT_SHAPES = {
    "IRA": (1, 24, 32),
    "wifi": (1, 2, 114),
    "imu": (1, 13, 17),
}

def safe_load_pickle(path):
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception as e:
        print(f"[load error] {path}: {e}")
        return None

def normalize_sample(data_dict):
    modality_data = data_dict.get("modality_data", {})
    for modality, default_shape in DEFAULT_SHAPES.items():
        node_list = modality_data.get(modality)
        if not node_list:
            modality_data[modality] = [{"frames": np.zeros(default_shape, dtype=np.float32)}]
            continue
        normalized_nodes = []
        for node_item in node_list:
            if node_item is None:
                normalized_nodes.append({"frames": np.zeros(default_shape, dtype=np.float32)})
                continue
            normalized_node = dict(node_item)
            frames = np.asarray(node_item.get("frames", []))
            if frames.size == 0:
                frames = np.zeros(default_shape, dtype=np.float32)
            if modality == "wifi":
                frames = np.abs(frames)
            normalized_node["frames"] = frames.astype(np.float32)
            normalized_nodes.append(normalized_node)
        modality_data[modality] = normalized_nodes
    data_dict["modality_data"] = modality_data
    return data_dict

# Multimodal model classes (must match training)
class ModalEncoder(nn.Module):
    def __init__(self, input_shape, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        flat_dim = int(input_shape[0] * input_shape[1])
        self.proj = nn.Linear(flat_dim, hidden_size)
        self.lstm = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        b, t = x.shape[0], x.shape[1]
        x = x.view(b, t, -1)
        x = self.proj(x)
        x = self.dropout(x)
        _, (h_n, _) = self.lstm(x)
        return h_n[-1]

class MultiModalLSTMClassifier(nn.Module):
    def __init__(self, modality_shapes, hidden_size=128, num_layers=2, dropout=0.2, num_classes=5):
        super().__init__()
        self.modalities = list(modality_shapes.keys())
        self.encoders = nn.ModuleDict({
            m: ModalEncoder(modality_shapes[m], hidden_size, num_layers, dropout)
            for m in self.modalities
        })
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * len(self.modalities), hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, batch_modality_dict):
        feats = []
        for m in self.modalities:
            x = batch_modality_dict[m]
            if m == "wifi":
                x = x.abs()
            feats.append(self.encoders[m](x))
        fused = torch.cat(feats, dim=1)
        return self.classifier(fused)

# Load saved multi-model
modality_shapes = {"IRA": (24, 32), "wifi": (2, 114), "imu": (13, 17)}
multi_path = OUT_DIR / "multi_model.pth"
if not multi_path.exists():
    raise FileNotFoundError(f"Multi-model checkpoint not found: {multi_path}")

multi_model = MultiModalLSTMClassifier(modality_shapes, hidden_size=128, num_layers=2, dropout=0.2)
state = torch.load(multi_path, map_location=DEVICE)
multi_model.load_state_dict(state)
multi_model.to(DEVICE)
multi_model.eval()

# Prediction helper for a single file using all modalities
def predict_single_file_multi(model, file_path):
    data = safe_load_pickle(file_path)
    if data is None:
        return None
    data = normalize_sample(data)
    inputs = {}
    for m in ["IRA", "wifi", "imu"]:
        node_list = data["modality_data"].get(m, [])
        if not node_list or node_list[0] is None:
            arr = np.zeros(DEFAULT_SHAPES[m], dtype=np.float32)
        else:
            arr = np.asarray(node_list[0].get("frames", []))
            if arr.size == 0:
                arr = np.zeros(DEFAULT_SHAPES[m], dtype=np.float32)
        if m == "wifi":
            arr = np.abs(arr)
        t = torch.from_numpy(arr).unsqueeze(0).float().to(DEVICE)  # (1, T, ...)
        inputs[m] = t
    with torch.no_grad():
        logits = model(inputs)
        pred = int(logits.argmax(dim=1).item()) + 1
    return pred

# Read CSV rows
rows = []
with open(CSV_FILE, "r", newline="") as f:
    reader = csv.reader(f)
    header = next(reader, None)
    rows = list(reader)

# Write activity_multi.csv
def write_pred_csv_multi(model, out_name="activity_multi.csv"):
    out_path = Path(out_name)
    out_rows = []
    out_header = ["filename", "activity", "activity_id"]
    for row in rows:
        if len(row) < 3:
            row = row + [""] * (3 - len(row))
        fname = row[0]
        label = (row[2] or "").strip()
        if label == "" or label == "nan":
            p = DATA_DIR / fname
            pred = predict_single_file_multi(model, p)
            if pred is None:
                out_rows.append([fname, "", ""])
            else:
                act_name = {
                    "1": "sitting", "2": "walking", "3": "sleeping",
                    "4": "falldown", "5": "jumping"
                }.get(str(pred), "")
                out_rows.append([fname, act_name, str(pred)])
        else:
            out_rows.append(row)
    with open(out_path, "w", newline="") as wf:
        writer = csv.writer(wf)
        writer.writerow(out_header)
        writer.writerows(out_rows)
    print(f"Wrote {out_path} (multi)")

write_pred_csv_multi(multi_model, "activity_multi.csv")

Wrote activity_multi.csv (multi)


### Task 4: Report (40 pt)
Write a report of around 4 pages about your findings, runtime and results. Use the same latex template as in the individual project. Put down the group members' name, student ID and email in the report.

In the report, include all your processing steps (include the AI workflows), as well as the GenAI declaration. Introduce the contribution of each group member in the report. 

- Marking criteria: 
    - Clarity of the report (10 pt)
    - Justification of the model architecture choices (10 pt)
    - Runtime, results, and analysis (10 pt)
    - Processing pipelines, AI workflows, etc (10 pt)

## Submission requirements:
__Pack the following materials into a zip file:__
2. all your code and trained checkpoints
3. the predicted csv with the same format as the training csv provided. Specifically, we use such a mapping for the activity labels:
    ```
        - 1: sitting
        - 2: walking
        - 3: sleeping
        - 4: falling down
        - 5: jumping
    ```
    The csv we provide contains 3 columns, filename,activity,activity_id. For the masked test set, the last two columns are blank. You should fill in the blank fields in the csv provided with your predictions, and submit the csv file. __You need to submit 4 csv files, with name activity_wifi.csv, activity_imu.csv, activity_ira.csv, activity_multi.csv, to represent the corresponding modality used.__
4. the report (pdf file) generated from the latex template provided. You can modify the body.tex file to write your report. Include all your group members' name, student ID and email in the report. Also include the visualized images in the report.
5. a short README.md file to the code structure and important steps to reproduce your results. You should make sure that your code is compatible with the provided data files and metadata. Include the library requirements if you use any additional libraries, and indicate this in the README.md file.

Each group only need to submit one copy. 

## a simple code illustration
__pay attention to the collating, in real world applications, the measurements are never perfectly aligned. You do not need to exactly fix the logic here, but the frame lengths may vary so you need to choose a suitable strategy to get a consistent input.__
You typically do not need to pad an empty segment since we already cleaned the data, but if you need to do so, you can refer to the code here.

This is a version without using OctoSense. We will also provide OctoSense, a unified framework for deep wireless sensing, in the future. You can also choose to use OctoSense to build your model.

```python
default_shapes = {
    'IRA': (T, 24, 32), # Time_frames, H, W
    'wifi': (T, 2, 114), # Time_frame, links, subcarriers
    'imu': (T, 13, 17), # Time_frame, 13 axes, 17 IMU nodes on joints
}
```

In [10]:
dataset_path = "data_sources"
dataset_csv = "activity_masked.csv"

import os
import csv
import random
import pickle
import numpy as np
import torch
from pathlib import Path
from pandas import Timestamp

SEED = 42

def set_global_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

set_global_seed(SEED)

from torch.utils.data import dataset
class sample_dataset(dataset.Dataset):
    def __init__(self, dataset_path, read_len=None, dataset_csv=dataset_csv):
        self.dataset_path = Path(dataset_path)
        self.csv_path = dataset_csv
        self.read_csv(read_len)

    def read_csv(self, read_len=None):
        self.samples = []
        with open(self.csv_path, 'r', newline='') as f:
            reader = csv.reader(f)
            next(reader, None)  # skip header
            for row in reader:
                if len(row) < 3:
                    continue
                raw_id = (row[2] or '').strip().lower()
                if raw_id in ('', 'nan'):
                    continue
                self.samples.append(row)
        if read_len is not None and read_len < len(self.samples):
            selected_indices = random.sample(range(len(self.samples)), read_len)
            self.samples = [self.samples[i] for i in selected_indices]

    def _pad_or_fill_frames(self, frames, default_shape, use_abs=False):
        arr = np.asarray(frames)
        if arr.size == 0:
            return np.zeros(default_shape, dtype=np.float32)
        if use_abs:
            arr = np.abs(arr)
        return arr.astype(np.float32)

    def _normalize_sample(self, data_dict):
        # pad or fill missing modalities and frames with zeros
        modality_data = data_dict.get('modality_data', {})
        default_shapes = {
            'IRA': (1, 24, 32), # Time_frames, H, W
            'wifi': (1, 2, 114), # Time_frame, links, subcarriers
            'imu': (1, 13, 17), # Time_frame, 13 axes, 17 IMU nodes on joints
        }

        for modality, default_shape in default_shapes.items():
            node_list = modality_data.get(modality)
            if not node_list:
                modality_data[modality] = [{'frames': np.zeros(default_shape, dtype=np.float32)}]
                continue

            normalized_nodes = []
            for node_item in node_list:
                if node_item is None:
                    normalized_nodes.append({'frames': np.zeros(default_shape, dtype=np.float32)})
                    continue

                normalized_node = dict(node_item)
                normalized_node['frames'] = self._pad_or_fill_frames(
                    node_item.get('frames', []),
                    default_shape,
                    use_abs=(modality == 'wifi'),
                )
                normalized_nodes.append(normalized_node)

            modality_data[modality] = normalized_nodes

        data_dict['modality_data'] = modality_data
        return data_dict

    @staticmethod
    def collate_fn(batch):
        # Minimal batch-time padding over time dimension for IRA/wifi/imu
        # while preserving original sample keys/metadata
        modalities = ('IRA', 'wifi', 'imu')
        default_shapes = {
            'IRA': (1, 24, 32),
            'wifi': (1, 2, 114), # note: wifi has two receiving antennas, so 2 channels here
            'imu': (1, 13, 17),
        }

        raw_samples, labels = zip(*batch)
        samples = [dict(s) for s in raw_samples]

        per_modality = {m: [] for m in modalities}
        max_time = {m: 1 for m in modalities}

        for sample in samples:
            modality_data = sample.get('modality_data', {})
            for modality in modalities:
                node_list = modality_data.get(modality, [])
                if not node_list or node_list[0] is None:
                    frames = np.zeros(default_shapes[modality], dtype=np.float32)
                else:
                    frames = np.asarray(node_list[0].get('frames', []))
                    if frames.size == 0:
                        frames = np.zeros(default_shapes[modality], dtype=np.float32)
                    else:
                        if modality == 'wifi':
                            frames = np.abs(frames)
                        frames = frames.astype(np.float32)

                per_modality[modality].append(frames)
                max_time[modality] = max(max_time[modality], frames.shape[0])

        batch_modality_data = {}
        for modality in modalities:
            padded = []
            target_t = max_time[modality]
            for frames in per_modality[modality]:
                if frames.shape[0] < target_t:
                    pad_shape = (target_t - frames.shape[0], *frames.shape[1:])
                    pad = np.zeros(pad_shape, dtype=np.float32)
                    frames = np.concatenate([frames, pad], axis=0)
                padded.append(torch.from_numpy(frames))
            batch_modality_data[modality] = torch.stack(padded, dim=0)

        labels_tensor = torch.tensor([int(x) for x in labels], dtype=torch.long)
        user_ids = [s.get('user_id') for s in samples]

        # Preserve only metadata in batch['metadata'] (drop raw frame arrays)
        metadata_samples = []
        for sample in samples:
            sample_meta = {}
            for key, value in sample.items():
                if key != 'modality_data':
                    sample_meta[key] = value

            modality_meta = {}
            for modality, node_list in sample.get('modality_data', {}).items():
                node_meta_list = []
                for node_item in node_list:
                    if node_item is None:
                        node_meta_list.append(None)
                    else:
                        node_meta = {k: v for k, v in node_item.items() if k != 'frames'}
                        node_meta_list.append(node_meta)
                modality_meta[modality] = node_meta_list

            # Flatten modality metadata under sample_meta (no nested 'modality_data')
            sample_meta.update(modality_meta)
            metadata_samples.append(sample_meta)

        return {
            'metadata': metadata_samples,
            'user_id': user_ids,
            'modality_data': batch_modality_data,
            'label': labels_tensor,
        }

    def __len__(self):
        # Return the total number of samples in the dataset
        return len(self.samples)

    def __getitem__(self, idx):
        # Load and return a sample from the dataset at the given index
        data_pickle_path = self.samples[idx][0]
        data_pickle_path = Path(self.dataset_path) / data_pickle_path
        with open(data_pickle_path, 'rb') as f:
            data_dict = pickle.load(f)
        data_dict = self._normalize_sample(data_dict)
        data_label = self.samples[idx][2]
        return data_dict, data_label

In [12]:
dataset = sample_dataset(dataset_path, read_len=None)
print(len(dataset))  # Check the total number of samples before reading CSV
dataset = sample_dataset(dataset_path, read_len=50)
print(f"Total samples in the dataset: {len(dataset)}")

400
Total samples in the dataset: 50


In the following, we provide a simple code illustration for training and evaluating a single modality (IRA) classifier. We use a simple MLP as an example, and we average the frames (which is actually not a good strategy). 

In [3]:
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, input_dim, num_layers=3) -> None:
        super().__init__()
        self.num_layers = num_layers
        self.input_dim = input_dim
        self.linear = nn.Linear(input_dim, 512)  # Example linear layer
        self.relu = nn.ReLU()  # Example activation function
        self.logits_layer = nn.Linear(512, 5)  # Example output layer for 5 classes
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten the input
        x = self.linear(x)  # Pass through the linear layer
        x = self.relu(x)  # Apply activation function
        x = self.logits_layer(x)  # Get logits for classification
        return x

In [4]:
import torch.utils.data.dataloader as dataloader
import torch
from torch.utils.data import Subset

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_set = Subset(dataset, list(range(45)))  # Use the first 45 samples for training
train_dataloader = dataloader.DataLoader(
    train_set,
    batch_size=1,
    shuffle=True,
    collate_fn=sample_dataset.collate_fn,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)
valid_set = Subset(dataset, list(range(45, len(dataset))))  # Use the remaining samples for validation
valid_dataloader = dataloader.DataLoader(
    valid_set,
    batch_size=1,
    shuffle=False,
    collate_fn=sample_dataset.collate_fn,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)

sample_data = next(iter(train_dataloader))  # Get the first batch from the training dataloader
print(f"Sample data keys: {sample_data.keys()}")
sample_label = sample_data['label']  # Get the label for the first sample in the batch
print(f"Keys in the sample data: {sample_data.keys()}")
print(f"Sample label: {sample_label}")
print(f"WiFi data shape: {sample_data['modality_data']['wifi'].shape}")
print(f"IRA data shape: {sample_data['modality_data']['IRA'].shape}")
print(f"IMU data shape: {sample_data['modality_data']['imu'].shape}")

print(f"keys: {sample_data.keys()}")
print(f"metadata keys: {sample_data['metadata'][0].keys()}")
print(f"IMU timestamps: {sample_data['metadata'][0]['imu'][0]['timestamps'][0:5]}")
print(f"IRA timestamps: {[Timestamp(t) for t in sample_data['metadata'][0]['IRA'][0]['timestamps'][0:5]]}")
print(f"WiFi timestamps: {[Timestamp(t) for t in sample_data['metadata'][0]['wifi'][0]['timestamps'][0:5]]}")

Sample data keys: dict_keys(['metadata', 'user_id', 'modality_data', 'label'])
Keys in the sample data: dict_keys(['metadata', 'user_id', 'modality_data', 'label'])
Sample label: tensor([1])
WiFi data shape: torch.Size([1, 233, 2, 114])
IRA data shape: torch.Size([1, 29, 24, 32])
IMU data shape: torch.Size([1, 256, 13, 17])
keys: dict_keys(['metadata', 'user_id', 'modality_data', 'label'])
metadata keys: dict_keys(['user_id', 'imu', 'wifi', 'IRA'])
IMU timestamps: [Timestamp('2024-05-19 16:24:41.883333'), Timestamp('2024-05-19 16:24:41.900000'), Timestamp('2024-05-19 16:24:41.916667'), Timestamp('2024-05-19 16:24:41.933333'), Timestamp('2024-05-19 16:24:41.950000')]
IRA timestamps: [Timestamp('2024-05-19 16:24:41.972596'), Timestamp('2024-05-19 16:24:42.103996'), Timestamp('2024-05-19 16:24:42.240314'), Timestamp('2024-05-19 16:24:42.405424'), Timestamp('2024-05-19 16:24:42.543941')]
WiFi timestamps: [Timestamp('2024-05-19 16:24:41.923543'), Timestamp('2024-05-19 16:24:42.002225'), Tim

In [5]:
def train(model, dataloader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0.0
        total_count = 0

        for batch in dataloader:
            ira_data = batch['modality_data']['IRA']
            labels = batch['label'] - 1  # convert 1..5 -> 0..4

            # temporal mean pooling to fixed size [B, 24, 32]
            ira_data = ira_data.mean(dim=1)

            outputs = model(ira_data)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            bsz = labels.size(0)
            total_loss += loss.item() * bsz
            total_count += bsz
            
        avg_loss = total_loss / total_count if total_count > 0 else 0.0
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

    return model

def evaluate(model, dataloader):
    model.eval()
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for batch in dataloader:
            ira_data = batch['modality_data']['IRA']
            labels = batch['label'] - 1  # convert 1..5 -> 0..4

            # temporal mean pooling to fixed size [B, 24, 32]
            ira_data = ira_data.mean(dim=1)

            outputs = model(ira_data)
            preds = outputs.argmax(dim=1)

            total_correct += (preds == labels).sum().item()
            total_count += labels.size(0)

    avg_accuracy = total_correct / total_count if total_count > 0 else 0.0
    return avg_accuracy

In [6]:
set_global_seed(SEED)
criterion = nn.CrossEntropyLoss()
model = Model(input_dim=24 * 32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

trained_model = train(model, train_dataloader, criterion, optimizer, num_epochs=15)
acc = evaluate(trained_model, valid_dataloader)
print(f'Validation Accuracy: {acc:.4f}')

Epoch [1/15], Loss: 22.1364
Epoch [2/15], Loss: 2.8568
Epoch [3/15], Loss: 1.9828
Epoch [4/15], Loss: 1.6181
Epoch [5/15], Loss: 1.6123
Epoch [6/15], Loss: 1.6069
Epoch [7/15], Loss: 1.6014
Epoch [8/15], Loss: 1.5963
Epoch [9/15], Loss: 1.5916
Epoch [10/15], Loss: 1.5865
Epoch [11/15], Loss: 1.5822
Epoch [12/15], Loss: 1.5778
Epoch [13/15], Loss: 1.5741
Epoch [14/15], Loss: 1.5704
Epoch [15/15], Loss: 1.5666
Validation Accuracy: 0.0000


In [7]:
set_global_seed(SEED)
criterion = nn.CrossEntropyLoss()
model = Model(input_dim=24 * 32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)

trained_model = train(model, train_dataloader, criterion, optimizer, num_epochs=6)
acc = evaluate(trained_model, valid_dataloader)
print(f'Validation Accuracy: {acc:.4f}')

Epoch [1/6], Loss: 3.5115
Epoch [2/6], Loss: 1.6658
Epoch [3/6], Loss: 1.5964
Epoch [4/6], Loss: 1.5408
Epoch [5/6], Loss: 1.7306
Epoch [6/6], Loss: 1.5810
Validation Accuracy: 0.2000
